# Define Catalog and Schema

In [0]:
%sql
USE CATALOG github_lakehouse;
USE SCHEMA silver;

## Create the `pull_request_review_comment_events` Silver Table

This step creates the Delta table used to store cleaned and structured **PullRequestReviewCommentEvent** records in the Silver layer.

The table contains three groups of fields:

- **Common event fields** — General event information shared across GitHub event types, including event ID, timestamp, actor, and repository.
- **Bronze lineage fields** — Source metadata retained from the Bronze layer for traceability.
- **PullRequestReviewCommentEvent-specific fields** — Information about inline review comments associated with pull requests, including the comment action, pull request ID, pull request number, and comment ID.

Unlike regular `IssueCommentEvent` records, these events represent comments made during the **code review process**, typically comments attached to specific parts of a pull request diff.

The table is stored using **Delta Lake** (`USING DELTA`).

`CREATE TABLE IF NOT EXISTS` allows the notebook to be safely rerun without recreating an existing table.

> **Note:** This step only defines the destination table. Extraction, cleaning, transformation, and validation of review-comment records happen later.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.pull_request_review_comment_events (
    -- Common fields
    event_id         STRING,
    event_type       STRING,
    event_timestamp  TIMESTAMP,
    event_date       DATE,
    actor_id         BIGINT,
    actor_login      STRING,
    repo_id          BIGINT,
    repo_name        STRING,

    -- Event-specific fields
    comment_action  STRING,
    pr_id           BIGINT,
    pr_number       BIGINT,
    comment_id      BIGINT,

    -- Bronze lineage
    _source_file     STRING,
    _source_date     DATE,
    _ingested_at     TIMESTAMP
)
USING DELTA;